We are here experimenting with QWEN 2.5 model to see the results.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip -q install -U transformers accelerate
!pip -q install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 154.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.8 MB/s eta 0:00:00


In [2]:
# Import libraries
from datasets import Dataset
from pandas import to_pickle
import json
from re import compile
import os

In [3]:
from datasets import load_from_disk
import pandas as pd

# Load Clean ObliQA dataset
qa_train = load_from_disk('/content/drive/MyDrive/Colab Notebooks/Obliqa_new/train_dataset_clean')
qa_test = load_from_disk('/content/drive/MyDrive/Colab Notebooks/Obliqa_new/test_dataset_clean')
qa_eval = load_from_disk('/content/drive/MyDrive/Colab Notebooks/Obliqa_new/eval_dataset_clean')
print(len(qa_train))
print(len(qa_test))
print(len(qa_eval))


26883
3491
3733


In [4]:

# LOAD CORPUS
# ===============================================
import pickle
corpus_path = "/content/drive/MyDrive/Colab Notebooks/Obliqa_new/corpus_clean.pkl"
with open(corpus_path, "rb") as f:
    corpus = pickle.load(f)


if isinstance(corpus, dict):
    doc_ids = list(corpus.keys())
    corpus_list = list(corpus.values())
else:
    corpus_list = corpus
    doc_ids = list(range(len(corpus_list)))

print(f" Loaded corpus with {len(corpus_list)} passages")


 Loaded corpus with 13705 passages


# Retrival of Top K answers with Entail Hard Neg Tuned Model




In [5]:
import json

with open("/content/drive/MyDrive/Colab Notebooks/Obliqa_new/Hybrid_ETHNM_Top10.json") as f:
    top_k_results = json.load(f)

Tuekenv6 Answer generation

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 4-bit quantization config (same as yours, but fixed dtype)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

#  DeepSeek model
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

# important for generation stability
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRM

In [7]:
def build_prompt(question, passages):
    context = "\n\n".join([p["text"] for p in passages])

    prompt = (
        "You are a regulatory compliance assistant. Provide a detailed answer for the question that "
        "fully integrates all the obligations and best practices from the given passages. Ensure your "
        "response is cohesive and directly addresses the question. Synthesize the information from all "
        "passages into a single, unified answer.\n\n"

        "Here are some examples:\n\n"

        # 🔹 Example 1 (FULL obligation coverage)
        "Example 1:\n"
        "Question: What procedures should a Recognised Body have in place to ensure proper handling of complaints?\n"
        "Context: A Recognised Body should acknowledge complaints promptly; conduct an objective, prompt and thorough investigation; "
        "provide a timely reply; inform the complainant of their right to apply to the complaints investigator; and keep adequate records.\n"
        "Answer: A Recognised Body should have arrangements to acknowledge complaints promptly, conduct an objective, prompt and thorough "
        "investigation, and provide a timely reply to the complainant. It should also inform the complainant of their right to apply to the "
        "complaints investigator if dissatisfied and maintain adequate records of complaints and investigations.\n\n"

        # 🔹 Example 2 (independence + recommendation handling)
        "Example 2:\n"
        "Question: How should a Recognised Body ensure that complaints are investigated fairly and that recommendations are considered?\n"
        "Context: Complaints must be investigated by an independent person. The investigator should have access to records, staff, and key individuals, "
        "and arrangements must exist for the Recognised Body to consider the investigator’s report and recommendations.\n"
        "Answer: A Recognised Body should ensure that complaints are investigated fairly and impartially by an independent person. It should provide the "
        "complaints investigator with access to relevant records, staff, and key individuals to support a thorough investigation. The Recognised Body should "
        "also have arrangements to receive and consider the investigator’s report and recommendations to ensure appropriate action is taken.\n\n"

        # 🔹 Example 3 (NO hallucination / strict grounding)
        "Example 3:\n"
        "Question: What is the required retention period for records of disclosures of conflicts of interest?\n"
        "Context: A Recognised Body should keep records of disclosures of conflicts of interest and the steps taken to handle them.\n"
        "Answer: The regulations require that records of disclosures of conflicts of interest and the steps taken to handle them are maintained. "
        "However, no specific retention period is defined in the provided provisions.\n\n"

        "Now answer the following question:\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )

    return prompt

In [8]:
# Prepare qa_eval_dict (anchor_id → gold answer) qa_eval_dict → map of qid to gold answer (we’ll prepare this too)

qa_eval_dict = {item["question_id"]: item["passage"] for item in qa_eval}

In [9]:
from transformers import pipeline
import json

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

generated_answers = {}
count = 0

for qid, entry in list(top_k_results.items())[:150]:
    question = entry["question"]
    passages = entry.get("filtered_passages", [])

    if not passages:
        continue  # skip if no filtered context

    prompt = build_prompt(question, passages)

    response = generator(prompt,max_new_tokens=512,do_sample=False,return_full_text=False)[0]["generated_text"]

    answer_text = response.split("Answer:")[-1].strip()

    generated_answers[qid] = {
        "question": question,
        "generated_answer": answer_text,
        "expected_answer": qa_eval_dict.get(qid, ""),
        "context_passages": passages
    }

    count += 1
    if count % 10 == 0:
        print(f"Generated answers for {count} questions...")

# Save to file for RePAS evaluation
output_path = "/content/drive/MyDrive/Colab Notebooks/Obliqa_new/DeepSeek_FewShotOrgPrompt_GenAns.json"
with open(output_path, "w") as f:
    json.dump(generated_answers, f, indent=2)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will t

Generated answers for 10 questions...


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Generated answers for 20 questions...


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

...

Generated answers for 130 questions...


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Generated answers for 140 questions...


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Generated answers for 150 questions...


In [ ]:
#first_key = next(iter(generated_answers))
#print(generated_answers[first_key])

{'question': 'What procedures should a Recognised Body have in place to ensure that a complainant receives a timely reply following the initial investigation of their complaint?', 'generated_answer': "A Recognised Body should have procedures in place to acknowledge complaints promptly, conduct an objective, prompt, and thorough initial investigation, and provide a timely reply to the complainant after the initial investigation. The procedures should also ensure that adequate records of complaints and investigations are kept. Additionally, the complainant should be informed of their right to apply to the Recognised Body's complaints investigator. Procedures should be documented to ensure the existence of these procedures is brought to the attention of potential complainants.", 'expected_answer': '', 'context_passages': [{'doc_id': '10-2.13.3.Guidance.2.', 'text': "2.13.3.Guidance.2. When determining whether it has effective arrangements for the investigation and resolution of complaints

In [10]:
import json

save_path = "/content/drive/MyDrive/Colab Notebooks/Obliqa_new/DeepSeek_FewShotOrgPrompt_GenAns.json"
with open(save_path, "r") as f:
    qwen_outputs = json.load(f)

print(f"Total generated answers so far: {len(qwen_outputs)}")




Total generated answers so far: 150


# Repass Score calaculation for answers generated

In [11]:
import json

# Load original qwen-style output
with open("/content/drive/MyDrive/Colab Notebooks/Obliqa_new/DeepSeek_FewShotOrgPrompt_GenAns.json", "r") as f:
    raw_data = json.load(f)

converted_data = []

for uuid, content in raw_data.items():
    question_id = uuid
    passages = [p["text"] for p in content.get("context_passages", [])]
    answer = content.get("generated_answer", "")

    # Skip empty answers just like the original script does
    if not answer.strip():
        continue

    converted_data.append({
        "QuestionID": question_id,
        "RetrievedPassages": passages,
        "Answer": answer
    })

# Save to file
with open("/content/drive/MyDrive/Colab Notebooks/Obliqa_new/DeepSeek_FewShotOrgPrompt_FormatAns.json", "w") as f:
    json.dump(converted_data, f, indent=2)

In [12]:
import sys
sys.argv = [
    "",  # program name placeholder
    "--input_file", "/content/drive/MyDrive/Colab Notebooks/Obliqa_new/DeepSeek_FewShotOrgPrompt_FormatAns.json",
    "--group_method_name", "/content/drive/MyDrive/Colab Notebooks/Obliqa_new/DeepseekFewShotOrgPrompt_topk10"
]

In [13]:
# Repass score calculation

import json
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from nltk.tokenize import sent_tokenize as sent_tokenize_uncached
import nltk
from functools import cache
import tqdm
import os
import csv

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)



# Set up device for torch operations
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Correct path to the trained model and tokenizer
model_name = '/content/drive/MyDrive/Colab Notebooks/eval/RePAS/models/obligation-classifier-legalbert'

# Load the tokenizer and model for obligation detection
obligation_tokenizer = AutoTokenizer.from_pretrained(model_name)
obligation_model = AutoModelForSequenceClassification.from_pretrained(model_name)
obligation_model.to(device)
obligation_model.eval()


use_cuda = torch.cuda.is_available()
pipe_device = 0 if use_cuda else -1
coverage_nli_model = pipeline(
    "text-classification",
    model="microsoft/deberta-large-mnli",
    device=pipe_device,
    return_all_scores=True,
    truncation=True
)



# Load NLI model and tokenizer for obligation coverage using Microsoft's model
#coverage_nli_model = pipeline("text-classification", model="microsoft/deberta-large-mnli", device=device)

# Load NLI model and tokenizer for entailment and contradiction checks
nli_tokenizer = AutoTokenizer.from_pretrained('cross-encoder/nli-deberta-v3-xsmall')
nli_model = AutoModelForSequenceClassification.from_pretrained('cross-encoder/nli-deberta-v3-xsmall')
nli_model.to(device)
nli_model.eval()

# Define a cached version of sentence tokenization
@cache
def sent_tokenize(passage: str):
    return sent_tokenize_uncached(passage)

def softmax(logits):
    e_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return e_logits / np.sum(e_logits, axis=1, keepdims=True)

def get_nli_probabilities(premises, hypotheses):
    features = nli_tokenizer(premises, hypotheses, padding=True, truncation=True, return_tensors="pt").to(device)
    nli_model.eval()
    with torch.no_grad():
        logits = nli_model(**features).logits.cpu().numpy()
    probabilities = softmax(logits)
    return probabilities

def get_nli_matrix(passages, answers):
    entailment_matrix = np.zeros((len(passages), len(answers)))
    contradiction_matrix = np.zeros((len(passages), len(answers)))

    batch_size = 16
    for i, pas in enumerate(tqdm.tqdm(passages)):
        for b in range(0, len(answers), batch_size):
            e = b + batch_size
            probs = get_nli_probabilities([pas] * len(answers[b:e]), answers[b:e])  # Get NLI probabilities
            entailment_matrix[i, b:e] = probs[:, 2]   # entailment
            contradiction_matrix[i, b:e] = probs[:, 0]
    return entailment_matrix, contradiction_matrix

def calculate_scores_from_matrix(nli_matrix, score_type='entailment'):
    if nli_matrix.size == 0:
        return 0.0
    return np.round(np.mean(np.max(nli_matrix, axis=0)), 5)

def classify_obligations(sentences):
    inputs = obligation_tokenizer(sentences, padding=True, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = obligation_model(**inputs).logits
    predictions = torch.argmax(logits, dim=1).cpu().numpy()
    return predictions

def calculate_obligation_coverage_score(passages, answers):
    # Filter obligation sentences from passages
    obligation_sentences_source = []
    for passage in passages:
        sentences = sent_tokenize(passage)
        is_obligation = classify_obligations(sentences)
        obligation_sentences_source.extend([sent for sent, label in zip(sentences, is_obligation) if label == 1])

    # Filter obligation sentences from answers
    obligation_sentences_answer = []
    for answer in answers:
        sentences = sent_tokenize(answer)
        is_obligation = classify_obligations(sentences)
        obligation_sentences_answer.extend([sent for sent, label in zip(sentences, is_obligation) if label == 1])

    # Calculate coverage based on NLI entailment
    covered_count = 0
    for obligation in obligation_sentences_source:
        best_ent = 0.0
        for answer_sentence in obligation_sentences_answer:
            scores = coverage_nli_model({"text": obligation, "text_pair": answer_sentence})

            # Normalize pipeline output
            if isinstance(scores, dict):
                scores = [scores]
            elif isinstance(scores, list) and len(scores) > 0 and isinstance(scores[0], list):
                scores = scores[0]
            ent = next(
                (float(d["score"]) for d in scores
                if d["label"].upper() in ("ENTAILMENT", "LABEL_2")),
                0.0
            )
            best_ent = max(best_ent, ent)
        if best_ent > 0.7:
            covered_count += 1

    return covered_count / len(obligation_sentences_source) if obligation_sentences_source else 1.0


def calculate_final_composite_score(passages, answers):
    passage_sentences = [sent for passage in passages for sent in sent_tokenize(passage)]
    answer_sentences = [sent for answer in answers for sent in sent_tokenize(answer)]
    entailment_matrix, contradiction_matrix = get_nli_matrix(passage_sentences, answer_sentences)
    entailment_score = calculate_scores_from_matrix(entailment_matrix, 'entailment')
    contradiction_score = calculate_scores_from_matrix(contradiction_matrix, 'contradiction')
    obligation_coverage_score = calculate_obligation_coverage_score(passages, answers)

    composite_score = (obligation_coverage_score + entailment_score - contradiction_score + 1) / 3
    return np.round(composite_score, 5), entailment_score, contradiction_score, obligation_coverage_score

def main(input_file_path, group_method_name):
    # Create a directory with the group_method_name in the data folder
    output_dir = f'./data/{group_method_name}'
    os.makedirs(output_dir, exist_ok=True)

    # Define the paths for result files
    output_file_csv = os.path.join(output_dir, 'results.csv')
    output_file_txt = os.path.join(output_dir, 'results.txt')

    processed_question_ids = set()
    composite_scores = []
    entailment_scores = []
    contradiction_scores = []
    obligation_coverage_scores = []

    # Check if the output CSV file already exists and read processed QuestionIDs
    if os.path.exists(output_file_csv):
        with open(output_file_csv, 'r') as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                processed_question_ids.add(row['QuestionID'])

    with open(input_file_path, 'r') as file:
        test_data = json.load(file)

    # Open the CSV file for appending results
    with open(output_file_csv, 'a', newline='') as csvfile:
        writer = csv.writer(csvfile)
        if not processed_question_ids:
            # Write the header if the file is empty or new
            writer.writerow(['QuestionID', 'entailment_score', 'contradiction_score', 'obligation_coverage_score', 'composite_score'])

        for item in tqdm.tqdm(test_data):
            question_id = item['QuestionID']

            # Skip if the QuestionID has already been processed
            if question_id in processed_question_ids:
                print(f"Skipping QuestionID {question_id}, already processed.")
                continue

            # Skip if the "Answer" is null or empty
            if not item.get('Answer') or not item['Answer'].strip():
                print(f"Skipping QuestionID {question_id}, no answer.")
                continue

            # Merge "RetrievedPassages" if it's a list
            if isinstance(item['RetrievedPassages'], list):
                item['RetrievedPassages'] = " ".join(item['RetrievedPassages'])

            passages = [item['RetrievedPassages']]
            answers = [item['Answer']]
            composite_score, entailment_score, contradiction_score, obligation_coverage_score = calculate_final_composite_score(passages, answers)

            # Append the scores to the lists
            composite_scores.append(composite_score)
            entailment_scores.append(entailment_score)
            contradiction_scores.append(contradiction_score)
            obligation_coverage_scores.append(obligation_coverage_score)

            # Write the result to the CSV file
            writer.writerow([question_id, entailment_score, contradiction_score, obligation_coverage_score, composite_score])

    # Calculate averages
    avg_entailment = np.mean(entailment_scores)
    avg_contradiction = np.mean(contradiction_scores)
    avg_obligation_coverage = np.mean(obligation_coverage_scores)
    avg_composite = np.mean(composite_scores)

    # Print and save results to a text file
    results = (
        f"Average Entailment Score: {avg_entailment}\n"
        f"Average Contradiction Score: {avg_contradiction}\n"
        f"Average Obligation Coverage Score: {avg_obligation_coverage}\n"
        f"Average Final Composite Score: {avg_composite}\n"
    )

    print(results)

    with open(output_file_txt, 'w') as txtfile:
        txtfile.write(results)

    print(f"Processing complete. Results saved to {output_dir}")

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Evaluate Obligation Coverage")
    parser.add_argument('--input_file', type=str, required=True, help='Path to the input JSON file')
    parser.add_argument('--group_method_name', type=str, required=True, help='Method name for grouping results')

    args = parser.parse_args()

    main(args.input_file, args.group_method_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 48, in spawn_con

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/283M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-xsmall
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 2/2 [00:00<00:00, 34.28it/s]

100%|██████████| 150/150 [12:34<00:00,  5.03s/it]

Average Entailment Score: 0.6052198000000001
Average Contradiction Score: 0.3353040666666666
Average Obligation Coverage Score: 0.8364754083195745
Average Final Composite Score: 0.7021306666666667

Processing complete. Results saved to ./data//content/drive/MyDrive/Colab Notebooks/Obliqa_new/DeepseekFewShotOrgPrompt_topk10
